In [2]:
# Capa Silver - BioTIME (local, desde Bronze DuckDB)
import duckdb
import os
import shutil
import time

start = time.time()

# ========== CONFIGURACIÓN DE RUTAS (nuevas ubicaciones) ==========
bronze_db_path = 'P:/proyecto_1/datos/bronze_output/biotime_bronze.duckdb'
silver_output_dir = 'P:/proyecto_1/datos/silver_output'
gold_input_dir = 'P:/proyecto_1/datos/gold_input'

# Crear carpetas de salida (si no existen)
os.makedirs(silver_output_dir, exist_ok=True)
os.makedirs(gold_input_dir, exist_ok=True)

# Conectar a DuckDB (en memoria)
conn = duckdb.connect()

conn.execute(f"ATTACH '{bronze_db_path}' AS bronze_db (READ_ONLY);")


# Crear esquema Silver
conn.execute("CREATE SCHEMA IF NOT EXISTS silver;")

# Crear tabla Silver limpiada (CTAS)

conn.execute("""
CREATE OR REPLACE TABLE silver.biotime_cleaned AS
SELECT
    d.STUDY_ID,
    d.ID_ALL_RAW_DATA,
    d.YEAR,
    d.MONTH,
    d.DAY,
    -- Corrección de abundancia
    CASE
        WHEN d.ABUNDANCE > 10000 THEN d.ABUNDANCE / 10000.0
        ELSE d.ABUNDANCE
    END AS ABUNDANCE,
    d.BIOMASS,
    d.LATITUDE,
    d.LONGITUDE,
    d.DEPTH,
    LOWER(TRIM(d.valid_name)) AS species_name,
    COALESCE(LOWER(TRIM(d.taxon)), 'unknown') AS taxon_group,
    m.REALM,
    m.HABITAT,
    m.CLIMATE,
    m.TAXA,
    -- Validaciones
    CASE
        WHEN d.ABUNDANCE > 10000 THEN (d.ABUNDANCE / 10000.0) IS NOT NULL AND (d.ABUNDANCE / 10000.0) > 0
        ELSE d.ABUNDANCE IS NOT NULL AND d.ABUNDANCE > 0
    END AS is_abundance_valid,
    (d.YEAR BETWEEN 1800 AND 2025) AS is_year_valid,
    (d.ABUNDANCE > 10000) AS abundance_correction_applied,
    CURRENT_TIMESTAMP AS silver_timestamp
FROM bronze_db.bronze.biotime_data d
LEFT JOIN bronze_db.bronze.biotime_metadata m ON d.STUDY_ID = m.STUDY_ID
WHERE d.YEAR IS NOT NULL AND d.valid_name IS NOT NULL;
""")
print(f"Tabla creada. Tiempo parcial: {time.time()-start:.2f} segundos")

# Validar
total_rows = conn.execute("SELECT COUNT(*) FROM silver.biotime_cleaned").fetchone()[0]
print(f"Filas totales en Silver: {total_rows}")

# Exportar a Parquet (en silver_output)
silver_parquet = os.path.join(silver_output_dir, 'silver_biotime_cleaned.parquet')
print("Exportando a Parquet...")
conn.execute(f"COPY silver.biotime_cleaned TO '{silver_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD);")
print(f"Parquet guardado: {silver_parquet}")

# Copiar a la carpeta gold_input (para que Gold lo consuma)
shutil.copy(silver_parquet, os.path.join(gold_input_dir, 'silver_biotime_cleaned.parquet'))
print(f"Archivo copiado a {gold_input_dir}")

conn.close()


Tabla creada. Tiempo parcial: 7.68 segundos
Filas totales en Silver: 11989233
Exportando a Parquet...
Parquet guardado: P:/proyecto_1/datos/silver_output\silver_biotime_cleaned.parquet
Archivo copiado a P:/proyecto_1/datos/gold_input
